## Зачем начинаем с этой игры

Возьмём крестики-нолики на доске 3×3. Здесь удобно разобрать весь метод на одной позиции. Опишем допустимые ходы, научимся оценивать худший ответ соперника и проверим простое правило выбора хода.

## Дерево партий и граф состояний

В дереве узел представляет историю ходов, а ребро — следующий ход. Одна и та же доска может встретиться в разных ветках.

В графе узел представляет состояние. Для этой игры достаточно записать содержимое клеток и чей сейчас ход. Если две истории приводят к одному такому состоянию, продолжения можно исследовать один раз.

Например, обозначим строки буквами a–c, а столбцы цифрами 1–3. Последовательности ходов X b1, O a1, X c3, O a2 и X c3, O a2, X b1, O a1 приводят к одной доске. В обоих случаях ход за X.

## Что гарантирует ход

Оценим исход для X. Победа равна +1, ничья — 0, поражение — −1. Терминальная позиция — та, в которой партия уже закончилась.

- На ходе X берём **максимум** оценок доступных продолжений.
- На ходе O берём **минимум** этих оценок для X.
- Если простое правило предписало X конкретный ход, берём оценку именно его продолжения.

Такой расчёт называется **минимаксом**. Ниже — маленький учебный граф с тремя ходами из старта. Метка `draw` появляется в двух ветках, но обозначает один узел. Граф ацикличен, то есть из узла нельзя вернуться к нему же по ходу игры.

In [1]:
from functools import lru_cache

successors = {
    "start": ("best", "safe", "risky"),
    "best": ("win-a", "win-b"),
    "safe": ("win-a", "draw"),
    "risky": ("draw", "loss"),
}
turn = {"start": "X", "best": "O", "safe": "O", "risky": "O"}
terminal_score = {"win-a": 1, "win-b": 1, "draw": 0, "loss": -1}
outcome = {-1: "поражение", 0: "ничья", 1: "победа"}


@lru_cache(maxsize=None)
def value(state):
    if state in terminal_score:
        return terminal_score[state]
    scores = [value(child) for child in successors[state]]
    return max(scores) if turn[state] == "X" else min(scores)


print("Лучшая гарантия:", outcome[value("start")])
for label, move in (
    ("Ход с гарантией победы", "best"),
    ("Осторожный ход", "safe"),
    ("Рискованный ход", "risky"),
):
    print(f"{label}: {outcome[value(move)]}")

Лучшая гарантия: победа
Ход с гарантией победы: победа
Осторожный ход: ничья
Рискованный ход: поражение


В учебном графе осторожный ход сохраняет ничью при любом ответе O, хотя другой ход гарантирует победу. У рискованного хода есть продолжение с ничьей, но O может выбрать продолжение с победой для себя.

## Контрпример на доске

**Гипотеза.** В позиции, где можно избежать проигрыша, выбор свободного центра гарантирует как минимум ничью.

| Строка | 1 | 2 | 3 |
|---|---|---|---|
| a | O | O | · |
| b | X | · | · |
| c | · | · | X |

Ход X. Точка обозначает пустую клетку. Правило предлагает ход **b2**. Тогда O отвечает **a3** и сразу выигрывает по верхней строке.

Из этой же позиции X может выиграть. После **X a3** сопернику нужно занять **b3**, иначе X следующим ходом завершит третий столбец. Если O сыграет **b3**, ход **X c1** создаст две угрозы — в **b2** по диагонали и в **c2** по нижней строке. За один ход O закроет только одну из них.

Один такой пример опровергает исходную гарантию. Следующая проверка — уточнить порядок условий в правиле и посмотреть, какие позиции оно всё ещё упускает.

## Следующий эксперимент

1. Определить формат позиции, допустимые ходы и признаки конца партии.
2. Построить граф достижимых состояний из пустой доски.
3. Сравнить минимакс с коротким правилом. Для равных вариантов заранее записать порядок выбора.
4. Найти отдельно упущенные победы и предотвратимые поражения.

[Перейти к плану исследования шашек](checkers.ipynb)